# Merging on Indexes

### Using indexes for merging  
Up to this point, merging has been done by matching columns between tables. In this lesson, the focus shifts to combining tables based on their indexes. In many datasets, the index itself represents a unique identifier, which makes it suitable for joining tables.

### Tables with an index  
Consider a movies table that was introduced earlier. Initially, it uses the default numeric index that increases automatically. In another version of the same table, the movie identifier is used as the index instead of being a regular column.

### Assigning an index  
There are several ways to define an index for a table. When data is loaded from a CSV file, one common approach is to specify the index directly during import. However, this lesson does not emphasize how to create an index, but rather how to merge tables once an index is already in place.

### Merging datasets using indexes  
Previously, the movies and taglines tables were combined using a shared identifier column with a left join. The same result can be achieved by performing the merge using the index, as long as the index represents that identifier.

### How index-based merging works  
The merge operation looks very similar to a column-based merge. The key difference is that the index name is provided instead of a column name. The merge function automatically handles whether the reference is a column or an index. The output remains the same, except the identifier now appears as the index.

### Working with MultiIndex tables  
Index-based merging also applies to tables with multiple index levels. In this case, both tables use a combination of movie ID and cast ID as their index. One table contains movies featuring a specific actor, while the other lists characters associated with movies. These tables can be merged using both index levels.

### Merging on a MultiIndex  
When merging MultiIndex tables, multiple index level names are provided, similar to merging on multiple columns. With an inner join, only rows where all index levels match in both tables are included in the result.

### Using different index names  
Sometimes, the index level names differ between the tables being merged. In such cases, separate arguments can be used to specify which index from each table should be matched.

### Combining left and right index references  
When merging tables with different index names, it is necessary to clearly indicate which index belongs to the left table and which belongs to the right table. Additional flags are used to signal that the merge should be performed using indexes rather than columns. These flags ensure that the merge operation correctly interprets and matches the index values.


## Prepare Data

In [12]:
# Import pandas library
import pandas as pd

# Read the file and set index
movies = pd.read_pickle("datasets/movies.p")
movies.set_index('id', inplace=True)
ratings = pd.read_csv("datasets/ratings.csv", index_col=['id'])
sequels = pd.read_csv("datasets/sequels.csv", index_col=['id'])
financials = pd.read_csv("datasets/financials.csv", index_col=['id'])

## Exercise: Index merge for movie ratings

In this exercise, you will strengthen your understanding of index-based merges by combining movie information with rating details. You are given two tables: one containing movie data and another containing movie ratings. The goal is to join these tables in a way that preserves every movie entry, while adding rating information only where it exists.

Both the **movies** and **ratings** tables are already available.

### Instructions
- Combine the movies and ratings tables using the movie identifier.
- Make sure all records from the movies table appear in the final result, even if a movie does not have a corresponding rating.
- Store the merged table in a variable named `movies_ratings`.

In [11]:
# Merge movies with ratings while keeping all movie records
movies_ratings = movies.merge(
    ratings,
    on='id',
    how='left'
)

# Display the first few rows of the merged table
movies_ratings.head()

,title,popularity,release_date,Unnamed: 0,vote_average,vote_count
id,,,,,,
257,Oliver Twist,20.415572,2005-09-23,667,6.7,274.0
14290,Better Luck Tomorrow,3.877036,2002-01-12,4634,6.5,27.0
38365,Grown Ups,38.864027,2010-06-24,520,6.0,1705.0
9672,Infamous,3.680896,2006-11-16,2778,6.4,60.0
12819,Alpha and Omega,12.300789,2010-09-17,2162,5.3,124.0


## Exercise: Do sequels earn more?

This exercise brings together several concepts from the chapter to analyze whether movie sequels outperform their original films in terms of revenue. You will work with two datasets: one containing sequel relationships and another with financial information.  

First, you will combine these datasets using the movie ID as the index, making sure every sequel entry is preserved even if some financial data is missing. Then, you will perform a self-join on the merged table to place each sequel alongside its original movie. After that, you will compute the revenue difference between the sequel and the original and organize the results to identify which sequels earned the most compared to their predecessors.

The **sequels** and **financials** tables are already available.

### Instructions  
1. Merge the sequels table with the financials table using the movie ID index, keeping all rows from the sequels table. Save the result as `sequels_fin`.
2. Join `sequels_fin` with itself using an inner join so that original movies and their sequels can be compared. Apply appropriate suffixes and save the result as `orig_seq`.
3. Create a new column that represents the revenue difference between the sequel and the original movie.
4. Select the relevant title and difference columns, sort the data by the revenue difference in descending order, and display the top results.

In [13]:
# Combine sequel data with financial information
sequels_fin = sequels.merge(
    financials,
    how='left',
    left_index=True,
    right_index=True
)

# Join the table to itself to align originals with their sequels
orig_seq = sequels_fin.merge(
    sequels_fin,
    how='inner',
    left_on='sequel',
    right_index=True,
    suffixes=('_org', '_seq')
)

# Compute the revenue difference
orig_seq['diff'] = orig_seq['revenue_seq'] - orig_seq['revenue_org']

# Keep only the relevant columns
titles_diff = orig_seq[['title_org', 'title_seq', 'diff']]

# Sort and show the top results
print(titles_diff.sort_values(by='diff', ascending=False).head())


               title_org        title_seq          diff
id                                                     
331    Jurassic Park III   Jurassic World  1.144748e+09
272        Batman Begins  The Dark Knight  6.303398e+08
10138         Iron Man 2       Iron Man 3  5.915067e+08
863          Toy Story 2      Toy Story 3  5.696028e+08
10764  Quantum of Solace          Skyfall  5.224703e+08
